In [1]:
import pandas as pd
import numpy as np
import preprocessing as pp
import features as f
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
%load_ext autoreload
%autoreload 2

In [2]:
df = pd.read_csv("MiningProcess_Flotation_Plant_Database.csv", decimal=",")

In [3]:
df = pp.clean_basic_data(df, date_col='date')

In [4]:
df.columns

Index(['date', 'pct_iron_feed', 'pct_silica_feed', 'starch_flow', 'amina_flow',
       'ore_pulp_flow', 'ore_pulp_ph', 'ore_pulp_density',
       'flotation_column_01_air_flow', 'flotation_column_02_air_flow',
       'flotation_column_03_air_flow', 'flotation_column_04_air_flow',
       'flotation_column_05_air_flow', 'flotation_column_06_air_flow',
       'flotation_column_07_air_flow', 'flotation_column_01_level',
       'flotation_column_02_level', 'flotation_column_03_level',
       'flotation_column_04_level', 'flotation_column_05_level',
       'flotation_column_06_level', 'flotation_column_07_level',
       'pct_iron_concentrate', 'pct_silica_concentrate'],
      dtype='str')

In [5]:
train, test = pp.split_data_chronologically(df, train_size = 0.8)

In [6]:
train_agg = train.groupby('date').agg(['mean', 'std', 'min', 'max'])
test_agg = test.groupby('date').agg(['mean', 'std', 'min', 'max'])

In [7]:
for d in [train_agg, test_agg]:
    d.columns = ['_'.join(col).lower().replace(' ', '_').replace('%', 'pct') for col in d.columns]

In [8]:
train_agg.columns

Index(['pct_iron_feed_mean', 'pct_iron_feed_std', 'pct_iron_feed_min',
       'pct_iron_feed_max', 'pct_silica_feed_mean', 'pct_silica_feed_std',
       'pct_silica_feed_min', 'pct_silica_feed_max', 'starch_flow_mean',
       'starch_flow_std', 'starch_flow_min', 'starch_flow_max',
       'amina_flow_mean', 'amina_flow_std', 'amina_flow_min', 'amina_flow_max',
       'ore_pulp_flow_mean', 'ore_pulp_flow_std', 'ore_pulp_flow_min',
       'ore_pulp_flow_max', 'ore_pulp_ph_mean', 'ore_pulp_ph_std',
       'ore_pulp_ph_min', 'ore_pulp_ph_max', 'ore_pulp_density_mean',
       'ore_pulp_density_std', 'ore_pulp_density_min', 'ore_pulp_density_max',
       'flotation_column_01_air_flow_mean', 'flotation_column_01_air_flow_std',
       'flotation_column_01_air_flow_min', 'flotation_column_01_air_flow_max',
       'flotation_column_02_air_flow_mean', 'flotation_column_02_air_flow_std',
       'flotation_column_02_air_flow_min', 'flotation_column_02_air_flow_max',
       'flotation_column_03_air_

In [9]:
train_agg
[train_agg['pct_iron_concentrate_mean'].autocorr(lag=i) for i in range(1, 25)]

[np.float64(0.7510332789700752),
 np.float64(0.6509251639434674),
 np.float64(0.5621004917417253),
 np.float64(0.4757454592824068),
 np.float64(0.4020435878783905),
 np.float64(0.34375583917753716),
 np.float64(0.2997772303625676),
 np.float64(0.2529741260655795),
 np.float64(0.23179433520356785),
 np.float64(0.21747419629490775),
 np.float64(0.20016450414202464),
 np.float64(0.19394294703131182),
 np.float64(0.18280297359092587),
 np.float64(0.17645950117315387),
 np.float64(0.1729005393625541),
 np.float64(0.15513974410383508),
 np.float64(0.16451319394683744),
 np.float64(0.16724174387277646),
 np.float64(0.1606248878781498),
 np.float64(0.15778797233347075),
 np.float64(0.15633488811462581),
 np.float64(0.15726155189890437),
 np.float64(0.15241322791746953),
 np.float64(0.16082737764969324)]

In [10]:
train_final = f.engineer_process_features(train_agg)
test_final = f.engineer_process_features(test_agg)

In [11]:
train_final = f.add_time_features(train_final)
test_final = f.add_time_features(test_final)

In [12]:
test_final = test_final.dropna()

In [13]:
train_final = train_final.dropna()

In [14]:
train_final.columns

Index(['pct_iron_feed_mean', 'pct_iron_feed_std', 'pct_iron_feed_min',
       'pct_iron_feed_max', 'pct_silica_feed_mean', 'pct_silica_feed_std',
       'pct_silica_feed_min', 'pct_silica_feed_max', 'starch_flow_mean',
       'starch_flow_std',
       ...
       'pct_iron_concentrate_mean_rolling_mean_3h',
       'pct_iron_concentrate_mean_rolling_std_3h',
       'pct_iron_concentrate_std_lag1', 'pct_iron_concentrate_std_lag3',
       'pct_iron_concentrate_std_rolling_mean_3h',
       'pct_iron_concentrate_std_rolling_std_3h', 'hour', 'day_of_week',
       'month', 'time_idx'],
      dtype='str', length=132)

In [15]:
X_train = train_final.drop(columns=["pct_iron_concentrate_mean", "pct_iron_concentrate_std", "pct_iron_concentrate_min","pct_iron_concentrate_max", "pct_iron_concentrate_mean_rolling_mean_3h", "pct_iron_concentrate_mean_rolling_std_3h", "pct_iron_concentrate_mean_lag1", "pct_iron_concentrate_mean_lag3"])
y_train = train_final["pct_iron_concentrate_mean"]

X_test= test_final.drop(columns=["pct_iron_concentrate_mean", "pct_iron_concentrate_std", "pct_iron_concentrate_min", "pct_iron_concentrate_max", "pct_iron_concentrate_mean_rolling_mean_3h", "pct_iron_concentrate_mean_rolling_std_3h", "pct_iron_concentrate_mean_lag1", "pct_iron_concentrate_mean_lag3"])
y_test = test_final["pct_iron_concentrate_mean"]

In [16]:
# 1. Initialize
scaler = StandardScaler()

# 2. Fit ONLY on X_train
scaler.fit(X_train)

# 3. Transform both (this turns them into NumPy arrays)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Convert back to DataFrame to keep column names.
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [21]:
# 1. Calculate correlations between all features and the target

correlations = train_final.corr()["pct_iron_concentrate_mean"].sort_values(ascending=False)

# 2. Print the top "Drivers" (Positive and Negative)
print("--- Top Drivers of Iron Concentration ---")
print(correlations.head(10))
print("\n--- Features that Decrease Iron ---")
print(correlations.tail(10))

--- Top Drivers of Iron Concentration ---
pct_iron_concentrate_mean                    1.000000
pct_iron_concentrate_min                     0.998147
pct_iron_concentrate_max                     0.998070
pct_iron_concentrate_mean_lag1               0.750600
pct_iron_concentrate_mean_rolling_mean_3h    0.726363
pct_iron_concentrate_mean_lag3               0.562100
flotation_column_05_level_mean               0.201447
ore_pulp_ph_mean_rolling_mean_3h             0.184022
flotation_column_05_level_min                0.182394
flotation_column_07_level_mean               0.182373
Name: pct_iron_concentrate_mean, dtype: float64

--- Features that Decrease Iron ---
pct_iron_concentrate_std                   -0.163998
pct_iron_concentrate_std_rolling_std_3h    -0.171675
flotation_column_04_air_flow_min           -0.202599
pct_iron_concentrate_std_rolling_mean_3h   -0.220818
pct_silica_concentrate_std                 -0.262841
pct_silica_concentrate_min                 -0.798376
pct_silica_conc

In [22]:
X_train_scaled.to_csv("X_train.csv", index=False)
X_test_scaled.to_csv("X_test.csv", index=False)
y_test.to_csv("y_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)